# HypotheSAEs Quickstart

This notebook demonstrates basic usage of HypotheSAEs on a sample of the Yelp review dataset.
By default we use `gpt-5.2` for hypothesis generation (interpreting neurons) and `gpt-5-mini` for text annotation.

Set `OPENAI_KEY_SAE` below for OpenAI-hosted requests. If you are using a local OpenAI-compatible endpoint (for example, vLLM server mode), set `OPENAI_BASE_URL`; the key is optional for non-OpenAI hosts.


In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ['OPENAI_KEY_SAE'] = '...'  # Replace with your OpenAI API key (or set from another env var)
os.environ['OPENAI_BASE_URL'] = 'http://0.0.0.0:8001/v1'  # Optional: local OpenAI-compatible endpoint

import numpy as np
import pandas as pd

from hypothesaes.quickstart import train_sae, interpret_sae, generate_hypotheses, evaluate_hypotheses
from hypothesaes.embedding import get_openai_embeddings, get_local_embeddings

INTERPRETER_MODEL = "openai/gpt-oss-120b"
ANNOTATOR_MODEL = "openai/gpt-oss-120b"

# Optional request kwargs forwarded directly to the Responses API.
# Keep empty by default; set only what you need at call time.
# Example: INTERPRET_LLM_KWARGS = {"reasoning_effort": "low"}
INTERPRET_LLM_KWARGS = {"reasoning_effort": "low"}
ANNOTATION_LLM_KWARGS = {"reasoning_effort": "low"}

N_WORKERS_ANNOTATION = 10  # Lower if hitting API rate limits


**Load data**

The dataset we're using here is a subset of 20K Yelp reviews, with 2K reviews used for validation (during SAE training). 

The target variable is the `stars` column, which is a rating between 1 and 5. We treat this as a regression task.

There are also 2K reviews used for holdout evaluation, which we'll use at the end of the notebook.

In [2]:
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"
full_df = pd.read_csv(os.path.join(prefix, 'data', 'vast_train.csv'))

In [3]:
import random
rng = random.Random(0)
indices = list(range(len(full_df)))
rng.shuffle(indices)

In [4]:
train_stop = int(len(full_df) * .75)
train_df = full_df.iloc[indices[:train_stop]]
val_df = full_df.iloc[indices[train_stop:]]

In [5]:
len(train_df), len(val_df), len(full_df)

(10107, 3370, 13477)

In [6]:
DOC_FIELD = 'post'
TOPIC_FIELD = 'new_topic'
train_texts, train_topics = list(train_df[DOC_FIELD]), list(train_df[TOPIC_FIELD])
val_texts, val_topics     = list(val_df[DOC_FIELD])  , list(val_df[TOPIC_FIELD])

In [7]:
from hypothesaes.sae import SparseAutoencoder
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
# embed_name = "google-bert/bert-base-cased"
embed_name = "nomic-ai/modernbert-embed-base"
class CrossSae(SparseAutoencoder):
    def __init__(self,
                 **kwargs):
        embedder = AutoModelForSequenceClassification.from_pretrained(embed_name).eval()
        embed_dim = embedder.config.hidden_size
        print(f"embed_dim={embed_dim}")
        super().__init__(input_dim=2 * embed_dim, **kwargs)
        self.embed_dim = embed_dim
        self.embedder: torch.nn.Module = embedder
        self.tokenizer = AutoTokenizer.from_pretrained(embed_name)
        for w in self.embedder.parameters():
            w.requires_grad_ = False

        self.film_gate = torch.nn.Sequential(
            torch.nn.Linear(embed_dim, embed_dim, bias=True),
            torch.nn.Tanh()
        )
        self.film_bias = torch.nn.Linear(embed_dim, embed_dim)
        # self.post_tran = torch.nn.Linear(embed_dim, 2 * embed_dim)

    @torch.inference_mode
    def preprocess(self, targets: list[str], text: list[str], batch_size: int = 16):
        if len(targets) != len(text):
            raise ValueError("targets and text must have the same length")
        if batch_size < 1:
            raise ValueError("batch_size must be positive")

        device = next(self.embedder.parameters()).device
        if not targets:
            return torch.empty((0, 2 * self.embed_dim), device=device)

        def embed_batch(items: list[str]):
            tokenized = self.tokenizer(
                items,
                padding=False,
                truncation=True,
                return_attention_mask=True,
            )
            sorted_indices = sorted(
                range(len(items)),
                key=lambda index: len(tokenized["input_ids"][index]),
            )
            dataloader = DataLoader(
                sorted_indices,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=lambda indices: self.tokenizer.pad(
                    {
                        key: [tokenized[key][index] for index in indices]
                        for key in tokenized
                    },
                    return_tensors="pt",
                ),
                pin_memory=device.type == "cuda",
            )

            sorted_embeddings = []
            for batch in tqdm(dataloader, desc="Embedding batches"):
                sorted_embeddings.extend(self._embed(
                    batch.to(device, non_blocking=device.type == "cuda")
                ))

            embeddings = [None] * len(items)
            for index, embedding in zip(sorted_indices, sorted_embeddings):
                embeddings[index] = embedding
            return torch.stack(embeddings)

        text_embeddings = embed_batch(list(text))
        target_embeddings = embed_batch(list(targets))
        return torch.cat((target_embeddings, text_embeddings), dim=-1).detach().cpu()

    def _embed(self, doc_ids):
        return self.embedder.base_model(**doc_ids).last_hidden_state[:, 0]

    def forward(self, x: torch.Tensor):
        A = x[..., :self.embed_dim]
        B = x[..., self.embed_dim:]
        mixed = self.film_gate(A) * B + self.film_bias(A)
        recon, info = super().forward( torch.cat([A, mixed], dim=1) )
        return recon, info


In [8]:
from hypothesaes.embedding import get_joint_stance_embeddings
train_embeddings = get_joint_stance_embeddings(train_topics, train_texts)
val_embeddings = get_joint_stance_embeddings(val_topics, val_texts)

Invalid model-index. Not loading eval results into CardData.
Invalid model-index. Not loading eval results into CardData.
Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Embedding target/post pairs:   0%|          | 0/1264 [00:00<?, ?it/s]

Invalid model-index. Not loading eval results into CardData.
Invalid model-index. Not loading eval results into CardData.
Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Embedding target/post pairs:   0%|          | 0/422 [00:00<?, ?it/s]

In [ ]:
# sae = CrossSae(m_total_neurons=256, k_active_neurons=8, prefix_lengths=[32, 256]).cuda()
# sae = CrossSae(m_total_neurons=64, k_active_neurons=4, prefix_lengths=[8, 64]).cuda()
# sae = CrossSae(m_total_neurons=128, k_active_neurons=4, prefix_lengths=[16, 128]).cuda()
# sae = CrossSae(m_total_neurons=128, k_active_neurons=8, prefix_lengths=[16, 128]).cuda()
# sae = CrossSae(m_total_neurons=128, k_active_neurons=16, prefix_lengths=[16, 128]).cuda()
# sae = CrossSae(m_total_neurons=128, k_active_neurons=32, prefix_lengths=[16, 128]).cuda()
# train_embeddings = sae.preprocess(train_topics, train_texts, batch_size=16)
# val_embeddings = sae.preprocess(val_topics, val_texts, batch_size=16)
# sae.fit(
#     train_embeddings,
#     val_embeddings,
# )

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: nomic-ai/modernbert-embed-base
Key               | Status  | 
------------------+---------+-
head.norm.weight  | MISSING | 
classifier.bias   | MISSING | 
classifier.weight | MISSING | 
head.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


embed_dim=768


In [9]:
sae = train_sae(embeddings=train_embeddings, val_embeddings=val_embeddings,
                M=256, K=8, matryoshka_prefix_lengths=[32, 256])


  0%|          | 0/100 [00:00<?, ?it/s]

**Interpret neurons**  

Interpret a random subset of neurons in the SAE to sanity-check that the learned features, and their interpretations, seem reasonable. We generate and print labels for `n_random_neurons` neurons, and we also print out the top-activating texts for each neuron.

In [10]:
texts_for_prompt = [f"Topic: {target} | Post: {text}" for target, text in zip(train_topics, train_texts)]

In [11]:
# This instruction will be included in the neuron interpretation prompt.
# The below instructions are specific to Yelp, but you can customize this for your task.
# If you don't pass in task-specific instructions, there is a generic instruction (see src/interpret_neurons.py);
# task-specific instructions are optional, but they help produce hypotheses at the desired level of specificity.

TASK_SPECIFIC_INSTRUCTIONS = """All of the below samples are (topic, post) pairs from an online forum.
Features should describe a specific aspect of how the post does (or does not) relate the topic.
Examples could be:
- 'praises another entity with positive associations to the topic'
- 'praises another entity with negative associations to the topic'
- 'criticizes another entity with negative associations to the topic'
- 'criticizes another entity with positive associations to the topic'

Focus on the relationship between the post and the topic. If it's possible to determine whether the feature applies by the post alone, you're doing it wrong."""

# Interpret random neurons
results = interpret_sae(
    texts=texts_for_prompt,
    embeddings=train_embeddings,
    sae=sae,
    n_random_neurons=5,
    print_examples_n=3,
    task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
    interpreter_model=INTERPRETER_MODEL,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
)


Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Activations shape: (10107, 256)


Generating interpretations:   0%|          | 0/5 [00:00<?, ?it/s]


Neuron 146 (1.2% active): mentions Bibi Netanyahu in the context of moving the U.S. embassy to Tel Aviv

Top activating examples:
1. Topic: embassy at tel aviv | Post: To reward Bibi Netanyahu for his aggressive and hostile actions by such a move would be criminal. It would make the removal of the embassy back to Tel Aviv by Trump's successor send a message we wouldn't want to send, but would find unavoidable if peace were to be achieved.
2. Topic: bib | Post: To reward Bibi Netanyahu for his aggressive and hostile actions by such a move would be criminal. It would make the removal of the embassy back to Tel Aviv by Trump's successor send a message we wouldn't want to send, but would find unavoidable if peace were to be achieved.
3. Topic: bibi netanyahu | Post: To reward Bibi Netanyahu for his aggressive and hostile actions by such a move would be criminal. It would make the removal of the embassy back to Tel Aviv by Trump's successor send a message we wouldn't want to send, but woul

**Generate hypotheses**

Generate hypotheses which are predictive of the target variable.

The `selection_method` parameter defines how we compute neuron predictiveness (see `src/select_neurons.py` for more details):
- "separation_score": E[target | top-activating examples] - E[target | zero-activating examples]
- "correlation": pearson(neuron activations, target variable)
- "lasso": select N nonzero features with an L1 regularized model

This cell outputs a dataframe with the following columns:
- `neuron_idx`: The index of the neuron in the SAE (if you're using multiple SAEs, this will be a global index across all of them).
- `source_sae`: The SAE that the neuron was selected from.
- `target_{selection_method}`: The predictiveness of the neuron for the target variable, using the selected `selection_method`.
- `interpretation`: The natural language interpretation of the neuron.
- `interp_fidelity_score`: The F1 fidelity score for how well the neuron's interpretation actually corresponds to its activation pattern.

In [9]:
selection_method = "correlation"
results = generate_hypotheses(
    texts=texts,
    labels=labels,
    embeddings=embeddings,
    sae=sae,
    cache_name=CACHE_NAME,
    selection_method=selection_method,
    n_selected_neurons=20,
    n_candidate_interpretations=1,
    task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
    interpreter_model=INTERPRETER_MODEL,
    annotator_model=ANNOTATOR_MODEL,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
    annotation_llm_kwargs=ANNOTATION_LLM_KWARGS,
    n_workers_annotation=N_WORKERS_ANNOTATION, # Please lower this parameter if you are running into API rate limits
)

print("\nMost predictive features of Yelp reviews:")
pd.set_option('display.max_colwidth', None)
display(results.sort_values(by=f"target_{selection_method}", ascending=False).round(3))
pd.reset_option('display.max_colwidth')


Embeddings shape: (20000, 768)


Computing activations (batchsize=16384):   0%|          | 0/2 [00:00<?, ?it/s]

Activations shape: (20000, 256)

Step 1: Selecting top 20 predictive neurons

Step 2: Interpreting selected neurons


Generating interpretations:   0%|          | 0/20 [00:00<?, ?it/s]


Step 3: Scoring Interpretations
Found 0 cached items; annotating 2000 uncached items


Scoring neuron interpretation fidelity (20 neurons; 1 candidate interps per neuron; 100 examples to score each…


Most predictive features of Yelp reviews:


,neuron_idx,target_correlation,interpretation,f1_fidelity_score
3,14,0.273,mentions great food,0.692
4,11,0.254,mentions the chef is a Master,0.039
7,20,0.235,mentions the restaurant is family owned,0.179
10,9,0.159,mentions vegan menu or vegan options,0.145
13,1,0.132,mentions great ambience,0.738
16,23,0.124,mentions pizza pickup kept warm,0.039
19,234,-0.119,mentions being disappointed,0.911
18,76,-0.120,mentions that the restaurant used to be great,0.750
17,7,-0.124,mentions being overcharged for food,0.734
15,61,-0.131,mentions manager yelling at employees,0.241


**Evaluate held-out generalization**

Finally, we evaluate whether these are good hypotheses by testing whether their natural language interpretations can predict the target variable.  

We compute annotations for each hypothesized concept on a holdout set (not seen during SAE training & feature selection).

After annotation, we output a dataframe with the following columns:
- `hypothesis`: The natural language hypothesis (which came from interpreting a predictive neuron in the SAE)
- `separation_score`: How much the target variable differs when the concept is present vs. absent (i.e., $E[Y\mid\text{concept} = 1] - E[Y\mid\text{concept} = 0]$).
- `separation_pvalue`: The t-test p-value of the null hypothesis that the separation score is 0 (i.e., the concept is not associated with the target variable).
- `regression_coef`: The coefficient of the concept in a multivariate linear regression of the target variable on all concepts.
- `regression_pval`: The p-value of the null hypothesis that the regression coefficient is 0.
- `feature_prevalence`: The fraction of examples that contain the concept.

Additionally, we output the evaluation metrics used in the paper:
- Significant hypotheses: the number of hypotheses that are significant in the multivariate regression at a specified significance level (default $0.1$) after Bonferroni correction. You can pass in a different significance level using the `corrected_pval_threshold` parameter.
- AUC or $R^2$: how well the hypotheses collectively predict the target variable in the multivariate regression.


In [10]:
holdout_df = pd.read_json(os.path.join(base_dir, "yelp-demo-holdout-2K.json"), lines=True)
holdout_texts = holdout_df['text'].tolist()
holdout_labels = holdout_df['stars'].values

metrics, evaluation_df = evaluate_hypotheses(
    hypotheses_df=results,
    texts=holdout_texts,
    labels=holdout_labels,
    cache_name=CACHE_NAME,
    annotator_model=ANNOTATOR_MODEL,
    annotation_llm_kwargs=ANNOTATION_LLM_KWARGS,
    n_workers_annotation=N_WORKERS_ANNOTATION, # Please lower this parameter if you are running into API rate limits
)

pd.set_option('display.max_colwidth', None)
display(evaluation_df.sort_values(by="separation_score", ascending=False).round(3))
pd.reset_option('display.max_colwidth')

print("\nHoldout Set Metrics:")
print(f"R² Score: {metrics['r2']:.3f}")
print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} " 
      f"(p < {metrics['Significant'][2]:.3e})")


Step 1: Annotating texts with 20 hypotheses
Found 0 cached items; annotating 40000 uncached items


Annotating:   0%|          | 0/40000 [00:00<?, ?it/s]

Step 2: Computing predictiveness of hypothesis annotations


/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/hypothesaes/evaluation.py:146: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  results = model.fit()
/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/hypothesaes/evaluation.py:118: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  _, p_value = ttest_ind(pos_vals, neg_vals)


,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence
3,mentions great food,1.921,0.000,1.203,0.000,0.681
13,mentions great ambience,0.777,0.000,0.281,0.000,0.141
6,mentions good burger,0.765,0.000,0.090,0.447,0.027
10,mentions vegan menu or vegan options,0.704,0.001,0.101,0.449,0.021
7,mentions the restaurant is family owned,0.359,0.279,-0.042,0.837,0.009
18,mentions that the restaurant used to be great,-0.748,0.000,-0.211,0.090,0.024
11,mentions missing ingredients in order,-1.068,0.000,-0.166,0.038,0.084
14,mentions dish differs from menu description,-1.110,0.000,-0.366,0.000,0.086
8,mentions wait times of 20+ minutes for food,-1.374,0.000,-0.339,0.000,0.046
17,mentions being overcharged for food,-1.455,0.000,-0.084,0.537,0.042



Holdout Set Metrics:
R² Score: 0.633
Significant hypotheses: 7/20 (p < 5.000e-03)
